In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 01: Data Exploration
# Storage access via Unity Catalog External Location 
# (no SAS tokens, no Spark config — fully managed identity-based access)
# ─────────────────────────────────────────────────────────────

storage_account_name = "staxidatakc250253104"

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path   = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("✓ Paths configured (auth via Unity Catalog External Location)")
print(f"  Bronze: {bronze_path}")
print(f"  Silver: {silver_path}")
print(f"  Gold:   {gold_path}")

✓ Paths configured (auth via Unity Catalog External Location)
  Bronze: abfss://bronze@staxidatakc250253104.dfs.core.windows.net
  Silver: abfss://silver@staxidatakc250253104.dfs.core.windows.net
  Gold:   abfss://gold@staxidatakc250253104.dfs.core.windows.net


In [0]:
# List the parquet files in bronze
files = dbutils.fs.ls(f"{bronze_path}/yellow_taxi/2023/")

print(f"Found {len(files)} files in bronze/yellow_taxi/2023/:")
print("-" * 70)
total_mb = 0
for f in files:
    size_mb = f.size / 1024 / 1024
    total_mb += size_mb
    print(f"  {f.name:<40} {size_mb:>8.1f} MB")
print("-" * 70)
print(f"{'TOTAL':<42} {total_mb:>8.1f} MB")

Found 12 files in bronze/yellow_taxi/2023/:
----------------------------------------------------------------------
  yellow_tripdata_2023-01.parquet              45.5 MB
  yellow_tripdata_2023-02.parquet              45.5 MB
  yellow_tripdata_2023-03.parquet              53.5 MB
  yellow_tripdata_2023-04.parquet              51.7 MB
  yellow_tripdata_2023-05.parquet              55.9 MB
  yellow_tripdata_2023-06.parquet              52.5 MB
  yellow_tripdata_2023-07.parquet              46.1 MB
  yellow_tripdata_2023-08.parquet              45.9 MB
  yellow_tripdata_2023-09.parquet              45.7 MB
  yellow_tripdata_2023-10.parquet              56.3 MB
  yellow_tripdata_2023-11.parquet              53.5 MB
  yellow_tripdata_2023-12.parquet              54.2 MB
----------------------------------------------------------------------
TOTAL                                         606.3 MB


In [0]:
from pyspark.sql.functions import col

# Load each file individually with explicit casting on the troublesome columns,
# then union them. This handles the INT64 vs DOUBLE inconsistency in NYC TLC data.

problem_cols = ["airport_fee", "passenger_count", "RatecodeID", 
                "PULocationID", "DOLocationID", "VendorID", "payment_type"]

dfs = []
for month in range(1, 13):
    month_str = f"{month:02d}"
    file_path = f"{bronze_path}/yellow_taxi/2023/yellow_tripdata_2023-{month_str}.parquet"
    
    df_month = spark.read.parquet(file_path)
    
    # Cast problem columns to a consistent type
    for c in problem_cols:
        if c in df_month.columns:
            df_month = df_month.withColumn(c, col(c).cast("double"))
    
    dfs.append(df_month)

# Union all 12 dataframes
from functools import reduce
df_raw = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

record_count = df_raw.count()
column_count = len(df_raw.columns)

print(f"✓ Data loaded with normalised schema")
print(f"  Records: {record_count:>15,}")
print(f"  Columns: {column_count:>15}")

✓ Data loaded with normalised schema
  Records:      38,310,226
  Columns:              19


In [0]:
# Print the schema to understand what columns we're working with
print("Schema of yellow_taxi 2023:")
print("=" * 70)
df_raw.printSchema()

Schema of yellow_taxi 2023:
root
 |-- VendorID: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: double (nullable = true)
 |-- DOLocationID: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [0]:
# Show first 10 rows in an interactive table
display(df_raw.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
2.0,2023-01-01T00:32:10.000,2023-01-01T00:40:36.000,1.0,0.97,1.0,N,161.0,141.0,2.0,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
2.0,2023-01-01T00:55:08.000,2023-01-01T01:01:27.000,1.0,1.1,1.0,N,43.0,237.0,1.0,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2.0,2023-01-01T00:25:04.000,2023-01-01T00:37:49.000,1.0,2.51,1.0,N,48.0,238.0,1.0,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0
1.0,2023-01-01T00:03:48.000,2023-01-01T00:13:25.000,0.0,1.9,1.0,N,138.0,7.0,1.0,12.1,7.25,0.5,0.0,0.0,1.0,20.85,0.0,1.25
2.0,2023-01-01T00:10:29.000,2023-01-01T00:21:19.000,1.0,1.43,1.0,N,107.0,79.0,1.0,11.4,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0
2.0,2023-01-01T00:50:34.000,2023-01-01T01:02:52.000,1.0,1.84,1.0,N,161.0,137.0,1.0,12.8,1.0,0.5,10.0,0.0,1.0,27.8,2.5,0.0
2.0,2023-01-01T00:09:22.000,2023-01-01T00:19:49.000,1.0,1.66,1.0,N,239.0,143.0,1.0,12.1,1.0,0.5,3.42,0.0,1.0,20.52,2.5,0.0
2.0,2023-01-01T00:27:12.000,2023-01-01T00:49:56.000,1.0,11.7,1.0,N,142.0,200.0,1.0,45.7,1.0,0.5,10.74,3.0,1.0,64.44,2.5,0.0
2.0,2023-01-01T00:21:44.000,2023-01-01T00:36:40.000,1.0,2.95,1.0,N,164.0,236.0,1.0,17.7,1.0,0.5,5.68,0.0,1.0,28.38,2.5,0.0
2.0,2023-01-01T00:39:42.000,2023-01-01T00:50:36.000,1.0,3.01,1.0,N,141.0,107.0,2.0,14.9,1.0,0.5,0.0,0.0,1.0,19.9,2.5,0.0


In [0]:
from pyspark.sql.functions import year, month, count

monthly_counts = (
    df_raw
    .withColumn("pickup_year", year("tpep_pickup_datetime"))
    .withColumn("pickup_month", month("tpep_pickup_datetime"))
    .groupBy("pickup_year", "pickup_month")
    .agg(count("*").alias("trip_count"))
    .orderBy("pickup_year", "pickup_month")
)

display(monthly_counts)

pickup_year,pickup_month,trip_count
2001,1,6
2002,12,11
2003,1,6
2008,12,23
2009,1,15
2014,11,1
2022,10,11
2022,12,25
2023,1,3066726
2023,2,2914003


In [0]:
from pyspark.sql.functions import col

total = df_raw.count()
print("DATA QUALITY SNAPSHOT")
print("=" * 70)
print(f"Total records: {total:,}\n")

print("Null/missing values in key columns:")
key_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", 
            "trip_distance", "fare_amount", "total_amount", "PULocationID", "DOLocationID"]

for c in key_cols:
    null_count = df_raw.filter(col(c).isNull()).count()
    pct = (null_count / total) * 100
    print(f"  {c:<30} {null_count:>10,}  ({pct:.2f}%)")

print()
print("Sanity-check anomalies:")
print(f"  Negative fare_amount:   {df_raw.filter(col('fare_amount') < 0).count():>10,}")
print(f"  Zero trip_distance:     {df_raw.filter(col('trip_distance') == 0).count():>10,}")
print(f"  Zero passenger_count:   {df_raw.filter(col('passenger_count') == 0).count():>10,}")
print(f"  Trip distance > 100 mi: {df_raw.filter(col('trip_distance') > 100).count():>10,}")
print(f"  Total amount > $500:    {df_raw.filter(col('total_amount') > 500).count():>10,}")

DATA QUALITY SNAPSHOT
Total records: 38,310,226

Null/missing values in key columns:
  tpep_pickup_datetime                    0  (0.00%)
  tpep_dropoff_datetime                   0  (0.00%)
  passenger_count                 1,309,356  (3.42%)
  trip_distance                           0  (0.00%)
  fare_amount                             0  (0.00%)
  total_amount                            0  (0.00%)
  PULocationID                            0  (0.00%)
  DOLocationID                            0  (0.00%)

Sanity-check anomalies:
  Negative fare_amount:      381,650
  Zero trip_distance:        773,457
  Zero passenger_count:      583,005
  Trip distance > 100 mi:      1,304
  Total amount > $500:           654
